<a href="https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![Vista previa: augmentaciones (HFlip, VFlip, Rot90) aplicadas consistentemente a imagen y máscara](https://raw.githubusercontent.com/apmontesp/Landslides_-Applied-ML-Course/main/docs/figures/nb02_preprocessing_preview.png)

*Vista previa: augmentaciones (HFlip, VFlip, Rot90) aplicadas consistentemente a imagen y máscara*

# Notebook 02: Preprocesamiento y Análisis de Datos (Landslide4Sense)
**Proyecto:** Detección de Deslizamientos mediante Inteligencia Artificial  
**Institución:** Universidad EAFIT  

Este notebook unifica la conexión a Drive, el análisis de balance de clases y el estudio de índices espectrales.

## 1. Conexión y Configuración de Rutas
Montamos Drive y localizamos los archivos .h5 de forma recursiva.

In [ ]:
from google.colab import drive
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

drive.mount('/content/drive')

base_path = Path('/content/drive/MyDrive/Landslide4Sense')
detectar_img = list(base_path.glob('**/TrainData/img'))

if detectar_img:
    train_img_dir = detectar_img[0]
    train_mask_dir = detectar_img[0].parent / "mask"
    img_list = sorted(list(train_img_dir.glob("*.h5")))
    mask_list = sorted(list(train_mask_dir.glob("*.h5")))
    print(f"✅ Éxito! Datos encontrados en: {train_img_dir}")
    print(f"Imágenes: {len(img_list)} | Máscaras: {len(mask_list)}")
else:
    print("❌ ERROR: No se encontró la estructura de carpetas.")

## 2. Análisis Visual Multiespectral
Visualización de bandas Sentinel-2 y capas geotécnicas.

In [ ]:
def load_h5(file_path):
    with h5py.File(file_path, 'r') as f:
        return np.array(f[list(f.keys())[0]])

if len(img_list) > 0:
    idx = np.random.randint(0, len(img_list))
    img = load_h5(img_list[idx])
    mask = load_h5(mask_list[idx])
    
    rgb = img[:, :, [3, 2, 1]]
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)
    
    fig, ax = plt.subplots(1, 4, figsize=(22, 5))
    ax[0].imshow(rgb); ax[0].set_title("Sentinel-2 (RGB)"); ax[0].axis('off')
    ax[1].imshow(img[:,:,12], cmap='terrain'); ax[1].set_title("Pendiente (Slope)"); ax[1].axis('off')
    ax[2].imshow(img[:,:,13], cmap='gist_earth'); ax[2].set_title("Elevación (DEM)"); ax[2].axis('off')
    ax[3].imshow(mask, cmap='magma'); ax[3].set_title("Zonas de Deslizamiento"); ax[3].axis('off')
    plt.show()

## 3. Estadísticas de Clases
Cálculo del desbalance para el reporte de tesis.

In [ ]:
def check_balance(limit=50):
    clase_0, clase_1 = 0, 0
    for i in range(min(limit, len(mask_list))):
        m = load_h5(mask_list[i])
        clase_1 += np.sum(m == 1)
        clase_0 += np.sum(m == 0)
    total = clase_0 + clase_1
    print(f"Análisis de balance (píxeles):")
    print(f"- Sin deslizamiento: {clase_0/total:.2%}")
    print(f"- Con deslizamiento: {clase_1/total:.2%}")

check_balance()

## 4. Análisis Avanzado: Índices Espectrales e Histogramas
Estudio de NDVI y rangos de bandas para optimizar el entrenamiento.

In [ ]:
def analyze_features(idx):
    img = load_h5(img_list[idx])
    red = img[:,:,3]
    nir = img[:,:,7]
    ndvi = (nir - red) / (nir + red + 1e-8)
    
    fig, ax = plt.subplots(1, 3, figsize=(18, 5))
    im1 = ax[0].imshow(ndvi, cmap='RdYlGn')
    ax[0].set_title("Índice de Vegetación (NDVI)")
    plt.colorbar(im1, ax=ax[0])
    
    ax[1].hist(img[:,:,12].ravel(), bins=50, color='brown', alpha=0.7)
    ax[1].set_title("Distribución de Pendientes")
    
    ax[2].boxplot([img[:,:,i].ravel() for i in range(4)], labels=['B1','B2','B3','B4'])
    ax[2].set_title("Rangos de Valores por Banda")
    plt.show()

if len(img_list) > 0:
    analyze_features(np.random.randint(0, len(img_list)))

### Conclusión del Preprocesamiento
1. **Desbalance:** Se confirma un desbalance severo (aprox 2.7% positivos). Se usará **Focal Loss**.
2. **NDVI:** El índice muestra una clara transición en zonas de suelo desnudo, lo cual es prometedor para el modelo.

---

## 5. Escalamiento con `sklearn` — Paradigma `fit` / `transform`

Al igual que en el preprocesamiento clásico de tablas (StandardScaler, MinMaxScaler), aplicamos el mismo paradigma a los 14 canales del dataset multispectral. La clave: **el scaler se ajusta solo con TrainData** y se aplica igual a Validation y Test.

In [ ]:
import numpy as np
import h5py
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from tqdm.auto import tqdm

# ── Cargar una muestra de patches para demostrar el paradigma ────────────────
N_DEMO = 100
patches_flat = []

for ip in tqdm(img_list[:N_DEMO], desc='Cargando patches'):
    with h5py.File(ip, 'r') as f:
        patch = f[list(f.keys())[0]][:].astype(np.float32)  # (128,128,14)
    # Flatten espacial: cada pixel como una fila, 14 columnas = 14 canales
    patches_flat.append(patch.reshape(-1, 14))  # (16384, 14)

X_demo = np.vstack(patches_flat)  # (N*16384, 14)
print(f'Forma de datos para scaler: {X_demo.shape}')
print(f'Antes del escalamiento — Canal 0: media={X_demo[:,0].mean():.3f}, std={X_demo[:,0].std():.3f}')

# ── fit solo con TrainData (igual que en el curso con X_train) ───────────────
scaler = StandardScaler()
scaler.fit(X_demo)   # aprende media y std de cada canal

X_scaled = scaler.transform(X_demo)
print(f'Después del escalamiento — Canal 0: media={X_scaled[:,0].mean():.4f}, std={X_scaled[:,0].std():.4f}')
print(f'\nMedias por canal (deben ser ~0):')
print(np.round(scaler.mean_, 4))
print(f'\nDesviaciones estándar aprendidas (scale_):')
print(np.round(scaler.scale_, 4))


### Paradigma `fit` / `transform` aplicado a parches satelitales

Igual que en el ejemplo de Spotify del curso:

```
scaler.fit(X_train)      # aprende μ y σ de cada canal en TrainData
X_train_sc = scaler.transform(X_train)   # aplica Z-score
X_val_sc   = scaler.transform(X_val)     # MISMO scaler, NO re-ajusta
X_test_sc  = scaler.transform(X_test)    # MISMO scaler
```

> **Error común:** hacer `fit_transform` en validación/test — introduce data leakage porque el scaler aprendería la distribución del set de evaluación.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

# Comparar StandardScaler vs MinMaxScaler en Canal 0 (S2-Azul)
ch0_raw = X_demo[:5000, 0]

std_sc  = StandardScaler().fit_transform(ch0_raw.reshape(-1,1)).flatten()
mm_sc   = MinMaxScaler().fit_transform(ch0_raw.reshape(-1,1)).flatten()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, data, title, color in zip(
    axes,
    [ch0_raw, std_sc, mm_sc],
    ['Raw (Ch0 S2-Azul)', 'StandardScaler (Z-score)', 'MinMaxScaler [0,1]'],
    ['steelblue', 'orange', 'green']
):
    ax.hist(data, bins=50, color=color, alpha=0.8, edgecolor='black', linewidth=0.4)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Valor')
    ax.set_ylabel('Frecuencia')
    ax.grid(alpha=0.3)
    stats = f'μ={data.mean():.3f}  σ={data.std():.3f}'
    ax.text(0.97, 0.95, stats, transform=ax.transAxes,
            ha='right', va='top', fontsize=9, color='black',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

plt.suptitle('Comparación de escaladores — Canal 0 (Sentinel-2 Azul)', fontsize=11)
plt.tight_layout()
plt.show()
print('Para imágenes satelitales se usa StandardScaler (Z-score):')
print('  - Preserva outliers informativos (anomalías espectrales)')
print('  - MinMaxScaler es sensible a valores extremos en SAR/DEM')


---

## 6. Imputación — Manejo de valores NaN/Inf

Los patches satelitales pueden contener píxeles sin dato (nubes, bordes del sensor). Usamos `SimpleImputer` con la misma lógica del curso.

In [ ]:
from sklearn.impute import SimpleImputer

# Verificar presencia de NaN/Inf en los patches cargados
n_nan = np.isnan(X_demo).sum()
n_inf = np.isinf(X_demo).sum()
print(f'Valores NaN  : {n_nan}')
print(f'Valores Inf  : {n_inf}')

if n_nan > 0 or n_inf > 0:
    # Reemplazar Inf por NaN primero
    X_clean = np.where(np.isinf(X_demo), np.nan, X_demo)
    imputer = SimpleImputer(strategy='median')
    X_imputed = imputer.fit_transform(X_clean)
    print(f'NaN despues de imputacion: {np.isnan(X_imputed).sum()}')
else:
    print('Dataset limpio — no se requiere imputacion')
    print('(Los patches Landslide4Sense ya vienen sin valores faltantes)')


---

## 7. Pipeline de Preprocesamiento

Encadenamos imputación + escalamiento en un `Pipeline` de sklearn, igual que en el ejemplo de California Housing del curso.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Pipeline: imputer → scaler (mismo patrón del curso)
preprocess_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

# fit con TrainData
preprocess_pipeline.fit(X_demo)
X_processed = preprocess_pipeline.transform(X_demo)

print('Pipeline aplicado correctamente:')
print(preprocess_pipeline)
print(f'\nForma entrada : {X_demo.shape}')
print(f'Forma salida  : {X_processed.shape}')
print(f'Media canal 0 : {X_processed[:,0].mean():.6f}  (debe ser ~0)')
print(f'Std  canal 0  : {X_processed[:,0].std():.6f}   (debe ser ~1)')

# Visualizar los 14 canales procesados
fig, axes = plt.subplots(2, 7, figsize=(18, 5))
CHANNEL_NAMES = [
    'S2-B2\nAzul','S2-B3\nVerde','S2-B4\nRojo','S2-B8\nNIR',
    'S2-B8A\nNIR-A','S2-B11\nSWIR1','S2-B12\nSWIR2',
    'S1-VV\nSAR','S1-VH\nSAR','DEM\nElev','DEM\nSlope',
    'S2-B5\nRE1','S2-B6\nRE2','S2-B7\nRE3'
]
for i, ax in enumerate(axes.flat):
    ax.hist(X_processed[:2000, i], bins=40, color='steelblue', alpha=0.8, edgecolor='black', linewidth=0.3)
    ax.set_title(CHANNEL_NAMES[i], fontsize=8)
    ax.set_xlabel('Z-score', fontsize=7)
    ax.tick_params(labelsize=6)
    ax.grid(alpha=0.3)

plt.suptitle('Distribución Z-score de los 14 canales (post pipeline)', fontsize=11)
plt.tight_layout()
plt.show()


---

## 8. Data Leakage — ¿Cuándo hacer las transformaciones?

Siguiendo el principio del curso sobre particiones:

| Transformación | ¿Antes o después de partir? | Razón |
|---------------|---------------------------|-------|
| Z-score (StandardScaler) | **Después** — fit solo en TrainData | Evitar leakage de μ/σ de val/test |
| Augmentación (flip, rot) | **Solo en TrainData** durante entrenamiento | Val/Test deben ser reales |
| Cálculo de índices (NDVI) | **Antes** — es solo una operación matemática | No usa estadísticas del dataset |
| Imputación (mediana) | **Después** — fit solo en TrainData | La mediana es una estadística |

> En nuestro caso: los patches `.h5` ya están partidos en `TrainData/ValidData/TestData` por los organizadores del dataset, por lo que el riesgo de leakage espacial ya está controlado.

---

## Resumen del Preprocesamiento

| Técnica | Herramienta | Aplicación en Landslide4Sense |
|---------|-------------|------------------------------|
| Escalamiento Z-score | `StandardScaler` | Por canal, fit en TrainData |
| Imputación | `SimpleImputer(median)` | Valores NaN/Inf en píxeles SAR |
| Pipeline | `sklearn.Pipeline` | Imputer → Scaler encadenados |
| Augmentación | PyTorch manual | HFlip, VFlip, Rot90 — solo TrainData |
| Índices espectrales | NumPy | NDVI, NDWI — antes de particionar |

Siguiente paso → `03_baseline_rf.ipynb`